# Carga de datos para trabajar con SQL

**Objetivo**: Cargar de forma controlada y segura los datos ya transformados a tablas de datos para trabajar posteriormente con SQL

In [4]:
import sqlite3
import pandas as pd
from pathlib import Path

# Directorio raíz del proyecto
raiz = Path.cwd().parent

# Archivos de entrada y base de datos

ruta_ipv = raiz / "data/processed/core/ipv.csv"

ruta_ipc = raiz / "data/processed/core/ipc.csv"

ruta_tasa = raiz / "data/processed/core/tasa.csv"

ruta_salarios = raiz / "data/processed/core/salarios.csv"


ruta_db = raiz / "data/vivienda.db"

# Leer los datos conservando los códigos territoriales como texto
df_ipv = pd.read_csv(
    ruta_ipv,
    dtype={"cod_territorio": "string"}
)

df_ipc = pd.read_csv(
    ruta_ipc,
    dtype={"cod_territorio": "string"}
)

df_tasa = pd.read_csv(
    ruta_tasa,
    dtype={"cod_territorio": "string"}
)

df_salarios = pd.read_csv(
    ruta_salarios,
    dtype={"cod_territorio": "string"}
)



### Tabla: Tasa

In [10]:
# Comprobar dimensiones y valores ausentes
print("Dimensiones:", df_tasa.shape)
print("\nValores nulos:")
print(df_tasa.isna().sum())

# Comprobar duplicados
clave = ["territorio", "anio", "trimestre"]

assert not df_tasa.duplicated(subset=clave).any()

# Comprobar claves completas
assert df_tasa[clave].notna().all().all()

# Comprobar validez de los valores
assert df_tasa["trimestre"].between(1, 4).all()
assert (df_tasa["valor_m2"].dropna() > 0).all()

# Comprobar dimensiones conocidas
assert len(df_tasa) == 2520
assert df_tasa["valor_m2"].isna().sum() == 73

print("\nValidación superada")

Dimensiones: (2520, 5)

Valores nulos:
cod_territorio    126
territorio          0
anio                0
trimestre           0
valor_m2           73
dtype: int64

Validación superada


In [11]:
# Conectar a la base de datos
conexion = sqlite3.connect(ruta_db)

try:
    # Iniciar una transacción
    conexion.execute("BEGIN")

    # Comprobar si la tabla ya contiene registros
    registros_actuales = conexion.execute(
        "SELECT COUNT(*) FROM tasa"
    ).fetchone()[0]

    if registros_actuales != 0:
        raise ValueError(
            "La tabla tasa ya contiene datos. "
            "Se cancela la importación."
        )

    # Preparar los registros para SQLite
    columnas = [
        "cod_territorio",
        "territorio",
        "anio",
        "trimestre",
        "valor_m2"
    ]

    datos = [
        tuple(fila)
        for fila in df_tasa[columnas]
        .astype(object)
        .where(pd.notna(df_tasa[columnas]), None)
        .itertuples(index=False, name=None)
    ]

    # Insertar registros
    conexion.executemany(
        """
        INSERT INTO tasa (
            cod_territorio,
            territorio,
            anio,
            trimestre,
            valor_m2
        )
        VALUES (?, ?, ?, ?, ?)
        """,
        datos
    )

    # Confirmar cambios
    conexion.commit()

    print(f"Importados {len(datos)} registros")

except Exception:
    # Deshacer todas las inserciones si hay un error
    conexion.rollback()
    raise

finally:
    conexion.close()

Importados 2520 registros


In [12]:
with sqlite3.connect(ruta_db) as conexion:
    resultado = pd.read_sql_query(
        "SELECT * FROM tasa",
        conexion
    )

assert len(resultado) == len(df_tasa)

assert resultado["valor_m2"].isna().sum() == 73

assert not resultado.duplicated(
    subset=["territorio", "anio", "trimestre"]
).any()

print("Importación verificada")

Importación verificada


### Tabla: IPV

In [ ]:
# Comprobar dimensiones y valores ausentes
print("Dimensiones:", df_ipv.shape)
print("\nValores nulos:")
print(df_ipv.isna().sum())

# Comprobar duplicados
clave = ["territorio", "anio", "trimestre", "tipo_vivienda"]

assert not df_ipv.duplicated(subset=clave).any()

# Comprobar claves completas
assert df_ipv[clave].notna().all().all()

# Comprobar validez de los valores
assert df_ipv["trimestre"].between(1, 4).all()
assert (df_ipv["indice_ipv"].dropna() > 0).all()

# Comprobar dimensiones conocidas
assert len(df_ipv) == 4680
assert df_ipv["indice_ipv"].isna().sum() == 312

print("\nValidación superada")

Dimensiones: (4680, 6)

Valores nulos:
cod_territorio    234
territorio          0
anio                0
trimestre           0
tipo_vivienda       0
indice_ipv        312
dtype: int64

Validación superada


In [15]:
# Conectar a la base de datos
conexion = sqlite3.connect(ruta_db)

try:
    # Iniciar una transacción
    conexion.execute("BEGIN")

    # Comprobar si la tabla ya contiene registros
    registros_actuales = conexion.execute(
        "SELECT COUNT(*) FROM ipv"
    ).fetchone()[0]

    if registros_actuales != 0:
        raise ValueError(
            "La tabla ipv ya contiene datos. "
            "Se cancela la importación."
        )

    # Preparar los registros para SQLite
    columnas = [
        "cod_territorio",
        "territorio",
        "anio",
        "trimestre",
        "tipo_vivienda",
        "indice_ipv"
    ]

    datos = [
        tuple(fila)
        for fila in df_ipv[columnas]
        .astype(object)
        .where(pd.notna(df_ipv[columnas]), None)
        .itertuples(index=False, name=None)
    ]

    # Insertar registros
    conexion.executemany(
        """
        INSERT INTO ipv (
            cod_territorio,
            territorio,
            anio,
            trimestre,
            tipo_vivienda,
            indice_ipv
        )
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        datos
    )

    # Confirmar cambios
    conexion.commit()

    print(f"Importados {len(datos)} registros")

except Exception:
    # Deshacer todas las inserciones si hay un error
    conexion.rollback()
    raise

finally:
    conexion.close()

Importados 4680 registros


In [16]:
with sqlite3.connect(ruta_db) as conexion:
    resultado = pd.read_sql_query(
        "SELECT * FROM ipv",
        conexion
    )

assert len(resultado) == len(df_ipv)

assert resultado["indice_ipv"].isna().sum() == 312

assert not resultado.duplicated(
    subset=["territorio", "anio", "trimestre", "tipo_vivienda"]
).any()

print("Importación verificada")

Importación verificada


### Tabla: IPC

In [20]:
# Comprobar dimensiones y valores ausentes
print("Dimensiones:", df_ipc.shape)
print("\nValores nulos:")
print(df_ipc.isna().sum())

# Comprobar duplicados
clave = ["territorio", "anio", "mes", "grupo_consumo"]

assert not df_ipc.duplicated(subset=clave).any()

# Comprobar claves completas
assert df_ipc[clave].notna().all().all()

# Comprobar validez de los valores
assert df_ipc["mes"].between(1, 12).all()
assert (df_ipc["indice_ipc"].dropna() > 0).all()

# Comprobar dimensiones conocidas
assert len(df_ipc) == 83160
assert df_ipc["indice_ipc"].isna().sum() == 0

print("\nValidación superada")

Dimensiones: (83160, 6)

Valores nulos:
cod_territorio    4158
territorio           0
anio                 0
mes                  0
grupo_consumo        0
indice_ipc           0
dtype: int64

Validación superada


In [21]:
# Conectar a la base de datos
conexion = sqlite3.connect(ruta_db)

try:
    # Iniciar una transacción
    conexion.execute("BEGIN")

    # Comprobar si la tabla ya contiene registros
    registros_actuales = conexion.execute(
        "SELECT COUNT(*) FROM ipc"
    ).fetchone()[0]

    if registros_actuales != 0:
        raise ValueError(
            "La tabla ipc ya contiene datos. "
            "Se cancela la importación."
        )

    # Preparar los registros para SQLite
    columnas = [
        "cod_territorio",
        "territorio",
        "anio",
        "mes",
        "grupo_consumo",
        "indice_ipc"
    ]

    datos = [
        tuple(fila)
        for fila in df_ipc[columnas]
        .astype(object)
        .where(pd.notna(df_ipc[columnas]), None)
        .itertuples(index=False, name=None)
    ]

    # Insertar registros
    conexion.executemany(
        """
        INSERT INTO ipc (
            cod_territorio,
            territorio,
            anio,
            mes,
            grupo_consumo,
            indice_ipc
        )
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        datos
    )

    # Confirmar cambios
    conexion.commit()

    print(f"Importados {len(datos)} registros")

except Exception:
    # Deshacer todas las inserciones si hay un error
    conexion.rollback()
    raise

finally:
    conexion.close()

Importados 83160 registros


In [23]:
with sqlite3.connect(ruta_db) as conexion:
    resultado = pd.read_sql_query(
        "SELECT * FROM ipc",
        conexion
    )

assert len(resultado) == len(df_ipc)

assert resultado["indice_ipc"].isna().sum() == 0

assert not resultado.duplicated(
    subset=["territorio", "anio", "mes", "grupo_consumo"]
).any()

print("Importación verificada")

Importación verificada


### Tabla: Salarios

In [26]:
# Comprobar dimensiones y valores ausentes
print("Dimensiones:", df_salarios.shape)
print("\nValores nulos:")
print(df_salarios.isna().sum())

# Comprobar duplicados
clave = ["territorio", "anio"]

assert not df_salarios.duplicated(subset=clave).any()

# Comprobar claves completas
assert df_salarios[clave].notna().all().all()

# Comprobar validez de los valores
assert (df_salarios["salario_medio"].dropna() > 0).all()

# Comprobar dimensiones conocidas
assert len(df_salarios) == 306
assert df_salarios["salario_medio"].isna().sum() == 0

print("\nValidación superada")

Dimensiones: (306, 4)

Valores nulos:
cod_territorio    17
territorio         0
anio               0
salario_medio      0
dtype: int64

Validación superada


In [27]:
# Conectar a la base de datos
conexion = sqlite3.connect(ruta_db)

try:
    # Iniciar una transacción
    conexion.execute("BEGIN")

    # Comprobar si la tabla ya contiene registros
    registros_actuales = conexion.execute(
        "SELECT COUNT(*) FROM salarios"
    ).fetchone()[0]

    if registros_actuales != 0:
        raise ValueError(
            "La tabla salarios ya contiene datos. "
            "Se cancela la importación."
        )

    # Preparar los registros para SQLite
    columnas = [
        "cod_territorio",
        "territorio",
        "anio",
        "salario_medio"
    ]

    datos = [
        tuple(fila)
        for fila in df_salarios[columnas]
        .astype(object)
        .where(pd.notna(df_salarios[columnas]), None)
        .itertuples(index=False, name=None)
    ]

    # Insertar registros
    conexion.executemany(
        """
        INSERT INTO salarios (
            cod_territorio,
            territorio,
            anio,
            salario_medio
        )
        VALUES (?, ?, ?, ?)
        """,
        datos
    )

    # Confirmar cambios
    conexion.commit()

    print(f"Importados {len(datos)} registros")

except Exception:
    # Deshacer todas las inserciones si hay un error
    conexion.rollback()
    raise

finally:
    conexion.close()

Importados 306 registros


In [28]:
with sqlite3.connect(ruta_db) as conexion:
    resultado = pd.read_sql_query(
        "SELECT * FROM salarios",
        conexion
    )

assert len(resultado) == len(df_salarios)

assert resultado["salario_medio"].isna().sum() == 0

assert not resultado.duplicated(
    subset=["territorio", "anio"]
).any()

print("Importación verificada")

Importación verificada
